# 03 — Modelling: Nightly-Price Prediction

**Project:** Property Investment Intelligence System (BIU DS23)
**Owner:** Idan Gilad — models and quantitative evaluation
**Status:** Template. No data loaded, no models trained, no results.

## Purpose

Train and compare models that predict a listing's **nightly asking price** from Inside Airbnb data, against a cheap baseline, on one shared held-out split.

## Inputs and outputs

| | Location |
| --- | --- |
| Input | Processed dataset from `02_preprocessing` in `data/processed/` |
| Output | Trained model artifacts and metadata in `models/` (excluded from Git) |
| Output | Held-out predictions for `04_evaluation` |

## Guardrails

- The target is nightly **asking** price, not acquisition value or realized revenue.
- Duplicate listing identities across snapshots must not appear on both sides of the split.
- All preprocessing is fitted on training data only.
- Every approach is evaluated on the **same** held-out cases.

## 1. Setup

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import time

import numpy as np
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

# TODO: pin versions in requirements.txt once the library set is agreed
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

In [ ]:
# Paths (notebook lives in notebooks/, so the repo root is one level up)
REPO_ROOT = Path.cwd().parent
DATA_DIR = REPO_ROOT / "data" / "processed"
MODELS_DIR = REPO_ROOT / "models"

# TODO: replace with the actual file produced by 02_preprocessing
DATA_FILE = DATA_DIR / "listings_processed.parquet"

# TODO: fill in from data/sources.md
CITY = "TODO"
CURRENCY = "TODO"
SNAPSHOT_DATE = "TODO"

## 2. Load processed data

Expects the cleaned output of `02_preprocessing`. Price should already be numeric (raw Inside Airbnb prices are strings like `"$1,234.00"`).

In [ ]:
if not DATA_FILE.exists():
    raise FileNotFoundError(
        f"{DATA_FILE} not found. Run 02_preprocessing first, "
        "or update DATA_FILE to the processed dataset."
    )

df = pd.read_parquet(DATA_FILE)  # or pd.read_csv if the team uses CSV
print(df.shape)
df.head()

## 3. Target and features

Column names below are placeholders based on the Inside Airbnb listings schema. **Confirm each against `data/data_dictionary.md`.**

Review every feature for leakage: anything derived from price, or only known after a booking, must be excluded.

In [ ]:
TARGET = "price"
ID_COL = "id"  # listing identity, used for the grouped split

NUMERIC_FEATURES = [
    "accommodates",
    "bedrooms",
    "beds",
    "bathrooms",
    "minimum_nights",
    "number_of_reviews",
    "review_scores_rating",
    "latitude",
    "longitude",
    # TODO: add engineered features from 02_preprocessing
]

CATEGORICAL_FEATURES = [
    "neighbourhood_cleansed",
    "room_type",
    "property_type",
]

FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES

missing = [c for c in FEATURES + [TARGET, ID_COL] if c not in df.columns]
assert not missing, f"Columns missing from dataset: {missing}"

In [ ]:
# Target distribution: prices are usually right-skewed with extreme outliers
df[TARGET].describe()

In [ ]:
# Decision: model log(price) and convert back for reporting?
# TODO: record the decision and reasoning in docs/
USE_LOG_TARGET = True

def to_model_target(y):
    return np.log1p(y) if USE_LOG_TARGET else y

def from_model_target(y):
    return np.expm1(y) if USE_LOG_TARGET else y

## 4. Train / test split

A **grouped** split by listing id so the same listing (appearing in several snapshots) never lands in both train and test.

This split is shared by every approach below and by `04_evaluation`. Save the test ids so the split can be reproduced exactly.

In [ ]:
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_SEED)
train_idx, test_idx = next(splitter.split(df, groups=df[ID_COL]))

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

# Sanity check: no listing identity on both sides
overlap = set(train_df[ID_COL]) & set(test_df[ID_COL])
assert not overlap, f"{len(overlap)} listing ids appear in both train and test"

X_train, y_train = train_df[FEATURES], train_df[TARGET]
X_test, y_test = test_df[FEATURES], test_df[TARGET]

print(f"train: {len(train_df):,}  test: {len(test_df):,}")

In [ ]:
# Persist the split for reproducibility and for 04_evaluation
# TODO: decide where split files live (e.g. evaluation/)
# pd.Series(test_df[ID_COL].unique()).to_csv(REPO_ROOT / "evaluation" / "test_ids.csv", index=False)

## 5. Baseline: training-set median by neighbourhood and room type

As defined in the README: median nightly price per (neighbourhood, room type) computed on **training data only**, with a global-median fallback for unseen combinations.

Any learned model must beat this to justify its cost.

In [ ]:
class MedianGroupBaseline:
    def __init__(self, group_cols):
        self.group_cols = group_cols

    def fit(self, X, y):
        data = X[self.group_cols].assign(_y=y.values)
        self.group_medians_ = data.groupby(self.group_cols)["_y"].median()
        self.global_median_ = float(np.median(y))
        return self

    def predict(self, X):
        keys = pd.MultiIndex.from_frame(X[self.group_cols])
        preds = self.group_medians_.reindex(keys).to_numpy()
        return np.where(np.isnan(preds), self.global_median_, preds)

# TODO: move to src/models/ once stable so other notebooks can import it
baseline = MedianGroupBaseline(["neighbourhood_cleansed", "room_type"]).fit(X_train, y_train)

## 6. Learned models

Preprocessing lives inside the pipeline, so it is fitted on training data only.

In [ ]:
numeric_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
])

categorical_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=10)),
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipe, NUMERIC_FEATURES),
    ("cat", categorical_pipe, CATEGORICAL_FEATURES),
])

In [ ]:
models = {
    "ridge": Pipeline([
        ("prep", preprocessor),
        ("model", Ridge(alpha=1.0)),
    ]),
    "hist_gbm": Pipeline([
        ("prep", preprocessor),
        ("model", HistGradientBoostingRegressor(random_state=RANDOM_SEED)),
    ]),
    # TODO: add candidates (e.g. LightGBM / XGBoost) if the team agrees on the dependency
}

for name, pipe in models.items():
    pipe.fit(X_train, to_model_target(y_train))
    print(f"fitted {name}")

### 6.1 Hyperparameter tuning (optional)

Tune with cross-validation **inside the training set only** (use `GroupKFold` on listing id). Never tune on the test set.

In [ ]:
# TODO: GridSearchCV / RandomizedSearchCV with GroupKFold(n_splits=5) on train_df[ID_COL]

## 7. Quick comparison on the held-out set

Headline metrics only. Detailed error analysis, groundedness and cost comparison belong in `04_evaluation`.
Metrics are reported in the original currency per night, not in log space.

In [ ]:
def evaluate(name, predict_fn, X, y_true):
    start = time.perf_counter()
    y_pred = predict_fn(X)
    elapsed_ms = (time.perf_counter() - start) * 1000 / len(X)
    return {
        "approach": name,
        f"MAE ({CURRENCY}/night)": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "ms per prediction": elapsed_ms,
    }

results = [evaluate("baseline_median", baseline.predict, X_test, y_test)]
for name, pipe in models.items():
    results.append(evaluate(name, lambda X, p=pipe: from_model_target(p.predict(X)), X_test, y_test))

results_df = pd.DataFrame(results).round(2)
results_df

## 8. Initial error checks

Where does the model fail? Check whether errors concentrate in particular neighbourhoods, room types or price ranges.

In [ ]:
best_name = "hist_gbm"  # TODO: pick based on results_df
best_model = models[best_name]

test_df["pred"] = from_model_target(best_model.predict(X_test))
test_df["abs_error"] = (test_df["pred"] - test_df[TARGET]).abs()

test_df.groupby("room_type")["abs_error"].agg(["mean", "median", "count"])

In [ ]:
test_df.groupby("neighbourhood_cleansed")["abs_error"].agg(["mean", "count"]) \
    .sort_values("mean", ascending=False).head(15)

In [ ]:
# TODO: residual vs actual plot, error by price decile, feature importance
# (save figures to reports/figures/)

## 9. Save model artifact and metadata

Artifacts go to `models/`, which is excluded from Git. The metadata JSON records how the model was produced so results can be reproduced.

In [ ]:
import joblib

MODELS_DIR.mkdir(parents=True, exist_ok=True)
version = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M")

artifact_path = MODELS_DIR / f"{best_name}_{version}.joblib"
joblib.dump(best_model, artifact_path)

metadata = {
    "model": best_name,
    "version": version,
    "city": CITY,
    "currency": CURRENCY,
    "snapshot_date": SNAPSHOT_DATE,
    "data_file": DATA_FILE.name,
    "target": TARGET,
    "log_target": USE_LOG_TARGET,
    "features": FEATURES,
    "random_seed": RANDOM_SEED,
    "n_train": len(train_df),
    "n_test": len(test_df),
    "metrics": results_df.to_dict(orient="records"),
}
(MODELS_DIR / f"{best_name}_{version}.json").write_text(json.dumps(metadata, indent=2, default=str))
print(f"saved {artifact_path.name}")

## 10. Open decisions and TODOs

- [ ] Confirm feature columns against `data/data_dictionary.md`
- [ ] Decide on log target and outlier handling (cap? exclude?) and document it
- [ ] Agree split file location with the team so `04_evaluation` reuses it
- [ ] Move `MedianGroupBaseline` and pipeline builders into `src/models/`
- [ ] Choose and pin modelling libraries in `requirements.txt`
- [ ] Hyperparameter tuning with grouped cross-validation
- [ ] Prediction uncertainty (e.g. quantile models) for the agent and investor output
- [ ] Fill in the README evaluation table once results exist